In [ ]:
import pandas as pd
import requests
import os
from datetime import datetime
from io import StringIO

In [ ]:
def download_csv_ads(station_id, ghcn_folder, start_date=None, end_date=None, datatypes=None):
    """
    Download GHCN-Daily data for a station using NOAA Access Data Service (ADS) API.

    Parameters
    ----------
    station_id : str
        GHCN station ID (e.g., 'USC00054736')
    ghcn_folder : str
        Folder to save CSV files
    start_date : str, optional
        Start date (YYYY-MM-DD). Defaults to '1800-01-01'
    end_date : str, optional
        End date (YYYY-MM-DD). Defaults to today
    datatypes : list of str, optional
        Data types to download (e.g., ['PRCP', 'TMAX', 'TMIN', 'SNOW']).
        Defaults to all 5.
    """
    # Defaults
    if start_date is None:
        start_date = "1800-01-01"
    if end_date is None:
        end_date = datetime.today().strftime("%Y-%m-%d")
    if datatypes is None:
        datatypes = ["PRCP", "TMAX", "TMIN", "SNOW"]

    os.makedirs(ghcn_folder, exist_ok=True)
    csv_path = os.path.join(ghcn_folder, f"{station_id}.csv")

    base_url = f"https://www.ncei.noaa.gov/access/services/data/v1"
    params = {
        "dataset": "daily-summaries",
        "stations": station_id,
        "startDate": start_date,
        "endDate": end_date,
        "dataTypes": ",".join(datatypes),
        "format": "csv",
        "units": "standard"  # returns T in F, PRCP/SNOW in inches
    }

    try:
        r = requests.get(base_url, params=params, timeout=60)
        if r.status_code == 200:
            df = pd.read_csv(StringIO(r.text))
            if df.empty:
                print(f"No data returned for {station_id}.")
                return
            df.to_csv(csv_path, index=False)
            print(f"Downloaded and saved {station_id}.csv with {len(df)} records.")
        else:
            print(f"Failed to download {station_id} (HTTP {r.status_code}): {r.text}")
    except requests.exceptions.RequestException as e:
        print(f"Request failed for {station_id}: {e}")


In [ ]:
station_id = 'USC00058022'
ghcn_folder = "fresh-data"
start_date = '2022-04-01'
end_date = None
datatypes = None
download_csv_ads(station_id, ghcn_folder, start_date, end_date, datatypes)
